# CAST Phase 3 — SERENGETI MAD-X Transfer (VALIDATION)



In [ ]:
# Environment setup
!pip install -q -U adapters accelerate datasets

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import adapters
print(f"adapters {adapters.__version__} ready")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.5/295.5 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
Mounted at /content/drive
adapters 1.3.0 ready


In [ ]:
# Configuration
import gc
import json
import os
import random
import shutil
import warnings

import adapters.composition as ac
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from adapters import AutoAdapterModel, SeqBnConfig, SeqBnInvConfig
from datasets import load_dataset
from sklearn.metrics import f1_score
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

SAVE_DIR = "/content/drive/MyDrive/EmotionDetection/CAST_checkpoints"
P3_DIR = f"{SAVE_DIR}/Phase3"
CACHE_DIR = f"{P3_DIR}/data_cache"
CKPT_DIR = f"{P3_DIR}/epoch_ckpt_serengeti"
ADAPTER_DIR = f"{P3_DIR}/adapters_serengeti"
PRED_DIR = f"{P3_DIR}/predictions"
MODEL_CACHE_DIR = f"{SAVE_DIR}/model_cache_serengeti"
LOCAL_MODEL_CACHE = "/content/hf_cache_local_serengeti"

for directory in [
    P3_DIR,
    CACHE_DIR,
    CKPT_DIR,
    ADAPTER_DIR,
    PRED_DIR,
    MODEL_CACHE_DIR,
    LOCAL_MODEL_CACHE,
]:
    os.makedirs(directory, exist_ok=True)

os.environ["HF_HOME"] = MODEL_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = f"{MODEL_CACHE_DIR}/hub"
os.environ["HF_DATASETS_CACHE"] = f"{MODEL_CACHE_DIR}/datasets"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EMOTION_ORDER = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
LANG_ORDER = ["eng", "hin", "rus", "hau", "kin", "sun", "yor", "vmw", "pcm"]
DATASET_HF_NAME = "brighter-dataset/BRIGHTER-emotion-categories"

LANGUAGES = {
    "eng": {"name": "English", "tier": 1, "family": "Indo-European", "genus": "Germanic"},
    "hin": {"name": "Hindi", "tier": 1, "family": "Indo-European", "genus": "Indo-Aryan"},
    "rus": {"name": "Russian", "tier": 1, "family": "Indo-European", "genus": "Slavic"},
    "hau": {"name": "Hausa", "tier": 2, "family": "Afroasiatic", "genus": "Chadic"},
    "kin": {"name": "Kinyarwanda", "tier": 2, "family": "Niger-Congo", "genus": "Bantu"},
    "sun": {"name": "Sundanese", "tier": 2, "family": "Austronesian", "genus": "Sundic"},
    "yor": {"name": "Yoruba", "tier": 3, "family": "Niger-Congo", "genus": "Volta-Niger"},
    "vmw": {"name": "Emakhuwa", "tier": 3, "family": "Niger-Congo", "genus": "Bantu"},
    "pcm": {"name": "Nigerian Pidgin", "tier": 3, "family": "Creole", "genus": "English-Lexifier"},
}
if list(LANGUAGES) != LANG_ORDER:
    raise ValueError("LANGUAGES must follow LANG_ORDER.")

TARGET_CODES = LANG_ORDER.copy()
SUPPORTED_SERENGETI_TARGETS = {"hau", "kin", "yor", "vmw", "pcm"}
THIS_TIER_TARGETS = [
    code for code in LANG_ORDER if code in SUPPORTED_SERENGETI_TARGETS
]
EXTRA_LANGS = {
    "deu": {"family": "Indo-European", "genus": "Germanic"},
    "swe": {"family": "Indo-European", "genus": "Germanic"},
    "afr": {"family": "Indo-European", "genus": "Germanic"},
    "swa": {"family": "Niger-Congo", "genus": "Bantu"},
    "ibo": {"family": "Niger-Congo", "genus": "Volta-Niger"},
}
PCM_LEXIFIER_C2 = ["eng", "deu", "swe", "afr"]
PCM_LEXIFIER_C3 = ["eng"]

MODEL_NAME = "UBC-NLP/serengeti-E250"
LR = 5e-5
NUM_EPOCHS = 10
BATCH_SIZE = 8
MAX_LENGTH = 256
PATIENCE = 3
REDUCTION_FACTOR = 16

ADAPTER_CONFIG_LANG = SeqBnInvConfig(reduction_factor=REDUCTION_FACTOR)
ADAPTER_CONFIG_TASK = SeqBnConfig(reduction_factor=REDUCTION_FACTOR)
TASK_ADAPTER_NAME = "emotion_task"
RESULTS_PATH = f"{P3_DIR}/phase3_serengeti_validation.json"

# False performs result validation and statistical analysis only.
RUN_MODEL_TRAINING = True

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"Targets: {THIS_TIER_TARGETS}")
print(f"Learning rate: {LR}")
print(f"Outputs: {P3_DIR}")

# ── Published benchmark reference and Phase 3 annotation constants ───────────
BENCHMARK_A = {
    "eng": 0.823, "hin": 0.926, "rus": 0.901, "hau": 0.751, "kin": 0.657,
    "sun": 0.550, "yor": 0.461, "vmw": 0.325, "pcm": 0.674,
}
BENCHMARK_C = {
    "eng": 0.797, "hin": 0.919, "rus": 0.906, "hau": 0.709, "kin": 0.519,
    "sun": 0.467, "yor": 0.359, "vmw": 0.210, "pcm": 0.674,
}
BENCHMARK_SRC_A = "SemEval-2025 Task 11, Table 5 (Track A)"
BENCHMARK_SRC_C = "SemEval-2025 Task 11, Table 7 (Track C)"
PHASE3_BENCHMARK_STATUS = "Context only: model score is validation; published reference is test"

Device: cuda
Model: UBC-NLP/serengeti-E250
Targets: ['hau', 'kin', 'yor', 'vmw', 'pcm']
Learning rate: 5e-05
Outputs: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase3


In [ ]:
# Family/genus maps and C1/C2/C3 pool definitions
FAMILY = {c: v["family"] for c, v in {**LANGUAGES, **EXTRA_LANGS}.items()}
GENUS  = {c: v["genus"]  for c, v in {**LANGUAGES, **EXTRA_LANGS}.items()}


def get_c1_pool(target: str) -> list:
    """All other BRIGHTER target languages."""
    return [c for c in TARGET_CODES if c != target]


def get_c2_pool(target: str) -> list:
    """Same language family (including extra langs). PCM: English lexifier sources."""
    if target == "pcm":
        return PCM_LEXIFIER_C2
    fam = FAMILY[target]
    return [c for c in FAMILY if FAMILY[c] == fam and c != target]


def get_c3_pool(target: str) -> list:
    """Narrowest genus-level pool. HAU collapses to C2 (only Chadic language).
    PCM: lexifier-based C3 (English only)."""
    if target == "pcm":
        return PCM_LEXIFIER_C3
    if target == "hau":
        return get_c2_pool(target)
    gen = GENUS[target]
    return [c for c in GENUS if GENUS[c] == gen and c != target]


print(f"{'Lang':6s} {'|C1|':>5s} {'|C2|':>5s} {'|C3|':>5s}  C2 pool")
print("-" * 70)
for code_ in THIS_TIER_TARGETS:
    c1, c2, c3 = get_c1_pool(code_), get_c2_pool(code_), get_c3_pool(code_)
    print(f"{code_:6s} {len(c1):5d} {len(c2):5d} {len(c3):5d}  {c2}")
# Hausa has no valid C2/C3 source pool for this backbone and is reported as N/A.

Lang    |C1|  |C2|  |C3|  C2 pool
----------------------------------------------------------------------
hau        8     0     0  []
kin        8     4     2  ['yor', 'vmw', 'swa', 'ibo']
yor        8     4     1  ['kin', 'vmw', 'swa', 'ibo']
vmw        8     4     2  ['kin', 'yor', 'swa', 'ibo']
pcm        8     4     1  ['eng', 'deu', 'swe', 'afr']


In [ ]:
# Data loading from the shared parquet cache
def get_emotion_cols(df: pd.DataFrame) -> list:
    return [e for e in EMOTION_ORDER if e in df.columns and df[e].fillna(0).sum() > 0]


def load_split(lang_code: str, split: str):
    cpath = f"{CACHE_DIR}/{lang_code}_{split}.parquet"
    if os.path.exists(cpath):
        return pd.read_parquet(cpath)
    try:
        ds = load_dataset(DATASET_HF_NAME, lang_code)
        split_key = split if split in ds else {"validation": "dev", "dev": "validation"}.get(split)
        if not split_key or split_key not in ds:
            return None
        df = ds[split_key].to_pandas()
        for e in EMOTION_ORDER:
            if e not in df.columns:
                df[e] = 0
        df.to_parquet(cpath)
        return df
    except Exception as exc:
        print(f"  Could not load {lang_code}/{split}: {exc}")
        return None


needed_extra = set()
for t in THIS_TIER_TARGETS:
    needed_extra |= set(get_c2_pool(t)) | set(get_c3_pool(t))
needed_extra -= set(TARGET_CODES)
LANGS_TO_LOAD = TARGET_CODES + sorted(needed_extra)

print(f"Loading {len(LANGS_TO_LOAD)} languages ...")
DATA = {}
for i, code_ in enumerate(LANGS_TO_LOAD, 1):
    is_target = code_ in TARGET_CODES
    splits_needed = ["train", "validation"]  # validation run: no test split needed
    DATA[code_] = {}
    for split in splits_needed:
        df = load_split(code_, split)
        if df is not None:
            DATA[code_][split] = df
    tag = "target" if is_target else "source "
    print(f"  [{i:2d}/{len(LANGS_TO_LOAD)}] {code_} ({tag}) -- {list(DATA[code_].keys())}")
print("Data loaded.")

Loading 14 languages ...
  [ 1/14] eng (target) -- ['train', 'validation']
  [ 2/14] hin (target) -- ['train', 'validation']
  [ 3/14] rus (target) -- ['train', 'validation']
  [ 4/14] hau (target) -- ['train', 'validation']
  [ 5/14] kin (target) -- ['train', 'validation']
  [ 6/14] sun (target) -- ['train', 'validation']
  [ 7/14] yor (target) -- ['train', 'validation']
  [ 8/14] vmw (target) -- ['train', 'validation']
  [ 9/14] pcm (target) -- ['train', 'validation']
  [10/14] afr (source ) -- ['train', 'validation']
  [11/14] deu (source ) -- ['train', 'validation']
  [12/14] ibo (source ) -- ['train', 'validation']
  [13/14] swa (source ) -- ['train', 'validation']
  [14/14] swe (source ) -- ['train', 'validation']
Data loaded.


In [ ]:
# MAD-X model, dataset, training, and evaluation
# SERENGETI is Electra-based: AutoAdapterModel, not XLMRobertaAdapterModel.

def _atomic_json_write(path: str, data) -> None:
    tmp = f"{path}.tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)
    os.sync()


def save_predictions(phase, lang, condition, y_true, y_pred, emotions):
    """Save per-sample predictions for significance tests and error analysis."""
    payload = {"emotions": list(emotions),
               "y_true": np.asarray(y_true).astype(int).tolist(),
               "y_pred": np.asarray(y_pred).astype(int).tolist()}
    path = f"{PRED_DIR}/{phase}_{lang}_{condition}.json"
    tmp = f"{path}.tmp"
    with open(tmp, "w") as f:
        json.dump(payload, f); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)
    os.sync()


class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, emotion_cols, max_len=MAX_LENGTH):
        self.texts   = df["text"].tolist()
        self.labels  = df[emotion_cols].fillna(0).astype(float).values
        self.tok     = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], max_length=self.max_len,
                       padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids":      enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels":         torch.tensor(self.labels[idx], dtype=torch.float)}


def macro_f1(y_true, y_pred, emotions):
    scores = {e: round(f1_score(y_true[:, i], y_pred[:, i], zero_division=0), 4)
              for i, e in enumerate(emotions)}
    scores["macro_f1"] = round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4)
    return scores


def compute_class_weights(train_df, emotion_cols):
    pos_counts = train_df[emotion_cols].fillna(0).sum()
    weights    = len(train_df) / (2 * pos_counts.clip(lower=1))
    return torch.tensor(weights.values, dtype=torch.float)


def build_madx_model(num_labels, lang_adapter_name, pretrained_lang_adapter_path=None):
    model = AutoAdapterModel.from_pretrained(MODEL_NAME, cache_dir=LOCAL_MODEL_CACHE)
    if pretrained_lang_adapter_path:
        model.load_adapter(pretrained_lang_adapter_path, config=ADAPTER_CONFIG_LANG,
                           load_as=lang_adapter_name)
    else:
        model.add_adapter(lang_adapter_name, config=ADAPTER_CONFIG_LANG)
    model.add_adapter(TASK_ADAPTER_NAME, config=ADAPTER_CONFIG_TASK)
    model.add_classification_head(TASK_ADAPTER_NAME, num_labels=num_labels, multilabel=True)
    stack = ac.Stack(lang_adapter_name, TASK_ADAPTER_NAME)
    model.set_active_adapters(stack)
    model.train_adapter(stack)
    return model.to(DEVICE)


def _adapter_head_state(model):
    return {k: v.cpu().clone() for k, v in model.state_dict().items()
            if ("adapters" in k or "invertible_adapters" in k
                or f"heads.{TASK_ADAPTER_NAME}" in k)}


def train_madx(model, train_df, dev_df, tokenizer, emotion_cols, ckpt_dir,
               epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE):
    gc.collect(); torch.cuda.empty_cache()

    train_dl = DataLoader(EmotionDataset(train_df, tokenizer, emotion_cols),
                          batch_size=batch_size, shuffle=True)
    dev_dl   = DataLoader(EmotionDataset(dev_df, tokenizer, emotion_cols),
                          batch_size=batch_size * 2)
    print(f"      train: {len(train_df)} ({len(train_dl)} batches/epoch)  dev: {len(dev_df)}")

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    opt   = AdamW(trainable_params, lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(
        opt, num_warmup_steps=int(0.1 * len(train_dl) * epochs),
        num_training_steps=len(train_dl) * epochs)

    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=compute_class_weights(train_df, emotion_cols).to(DEVICE))
    scaler = GradScaler()

    os.makedirs(ckpt_dir, exist_ok=True)
    state_path = f"{ckpt_dir}/state.json"
    last_path  = f"{ckpt_dir}/last_epoch.pt"
    best_path  = f"{ckpt_dir}/best.pt"
    opt_path   = f"{ckpt_dir}/opt.pt"
    sched_path = f"{ckpt_dir}/sched.pt"
    log_path   = f"{P3_DIR}/live_training_logs.txt"

    start_ep, best_f1, no_improve = 1, 0.0, 0
    if os.path.exists(state_path):
        try:
            with open(state_path) as f:
                st = json.load(f)
            start_ep, best_f1, no_improve = st["epoch"] + 1, st["best_f1"], st["no_improve"]
            _active_resume = model.active_adapters
            model.load_state_dict(torch.load(last_path, map_location=DEVICE), strict=False)
            if _active_resume is not None:
                model.set_active_adapters(_active_resume)
            opt.load_state_dict(torch.load(opt_path, map_location=DEVICE))
            sched.load_state_dict(torch.load(sched_path, map_location=DEVICE))
            print(f"      Resuming from epoch {start_ep} (best F1: {best_f1:.4f})")
        except (json.JSONDecodeError, KeyError, ValueError):
            print("      Corrupt checkpoint -- starting fresh")

    for ep in range(start_ep, epochs + 1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_dl, desc=f"      epoch {ep}/{epochs}", leave=False)
        for step, batch in enumerate(pbar, 1):
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbl  = batch["labels"].to(DEVICE)
            opt.zero_grad()
            with autocast():
                out  = model(input_ids=ids, attention_mask=mask, head=TASK_ADAPTER_NAME)
                loss = loss_fn(out.logits, lbl)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            running_loss += loss.item()
            pbar.set_postfix(loss=f"{running_loss/step:.4f}")

        dev_res = evaluate_madx(model, dev_df, tokenizer, emotion_cols,
                                batch_size=batch_size * 2)
        dev_f1  = dev_res["macro_f1"]
        print(f"      epoch {ep}/{epochs} -- loss: {running_loss/len(train_dl):.4f} -- dev F1: {dev_f1:.4f}")

        with open(log_path, "a") as lf:
            lf.write(f"[{os.path.basename(ckpt_dir)}] epoch {ep}/{epochs}"
                     f" -- loss: {running_loss/len(train_dl):.4f} -- dev F1: {dev_f1:.4f}\n")

        if dev_f1 > best_f1:
            best_f1, no_improve = dev_f1, 0
            torch.save(_adapter_head_state(model), best_path)
            os.sync()
        else:
            no_improve += 1

        torch.save(_adapter_head_state(model), last_path)
        torch.save(opt.state_dict(),   opt_path)
        torch.save(sched.state_dict(), sched_path)
        with open(state_path, "w") as f:
            json.dump({"epoch": ep, "best_f1": best_f1, "no_improve": no_improve}, f)
        os.sync()

        if no_improve >= patience:
            print(f"      Early stop ({patience} epochs without improvement)")
            break

    ckpt = best_path if os.path.exists(best_path) else last_path
    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE), strict=False)
        if ckpt == last_path:
            print("      No best checkpoint -- using last epoch weights")
    shutil.rmtree(ckpt_dir, ignore_errors=True)
    return model


@torch.no_grad()
def evaluate_madx(model, test_df, tokenizer, emotion_cols,
                  batch_size=BATCH_SIZE * 2, run_tag=None, adapter_setup=None):
    test_dl = DataLoader(EmotionDataset(test_df, tokenizer, emotion_cols),
                         batch_size=batch_size)
    model.eval()
    if adapter_setup is not None:
        model.set_active_adapters(adapter_setup)
    preds_list, true_list = [], []
    for batch in test_dl:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        out  = model(input_ids=ids, attention_mask=mask, head=TASK_ADAPTER_NAME)
        preds_list.append((torch.sigmoid(out.logits) > 0.5).int().cpu().numpy())
        true_list.append(batch["labels"].int().numpy())
    y_true_all, y_pred_all = np.vstack(true_list), np.vstack(preds_list)
    if run_tag is not None:
        save_predictions(run_tag[0], run_tag[1], run_tag[2],
                         y_true_all, y_pred_all, emotion_cols)
    return macro_f1(y_true_all, y_pred_all, emotion_cols)


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=LOCAL_MODEL_CACHE)
print("SERENGETI tokenizer loaded. MAD-X functions ready.")

def _run_is_genuinely_complete(target, run_key, pred_tag):
    """
    Returns (is_complete, reason).
    Checks three things in order:
      1. Entry exists in the results JSON
      2. Prediction file exists on disk
      3. Adapter weights (lang + task) exist on disk
    N/A entries (empty pool / aliased) skip weight checks — no adapters were saved.
    Call this instead of the bare `run_key in phase3_results[target]` check so that
    a lost adapter directory triggers a re-run rather than a silent skip.
    """
    entry = phase3_results.get(target, {}).get(run_key)
    if entry is None:
        return False, "not in results JSON"

    # N/A or aliased — no adapter weights were ever saved for these
    if isinstance(entry, dict) and entry.get("macro_f1") is None:
        return True, "N/A (no training run)"
    if isinstance(entry, dict) and "aliased_from" in entry:
        return True, "aliased — no separate weights"

    # Prediction file
    pred_path = f"{PRED_DIR}/{pred_tag}_{target}_{run_key}.json"
    if not os.path.exists(pred_path):
        return False, f"prediction file missing: {os.path.basename(pred_path)}"

    # Adapter weights - both lang and task subdirs must have a pytorch_adapter.bin
    adir = f"{ADAPTER_DIR}/{target}_{run_key}"
    for sub in ["lang", "task"]:
        if not os.path.exists(f"{adir}/{sub}/pytorch_adapter.bin"):
            return False, f"adapter weights missing: {target}_{run_key}/{sub}/"

    return True, "complete"

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

SERENGETI tokenizer loaded. MAD-X functions ready.


In [ ]:
# Emakhuwa language-adaptive pre-training
from transformers import DataCollatorForLanguageModeling

VMW_LAPT_ADAPTER_NAME = "lang_vmw_lapt_serengeti"
VMW_LAPT_ADAPTER_PATH = f"{P3_DIR}/{VMW_LAPT_ADAPTER_NAME}"
LAPT_EPOCHS, LAPT_BATCH_SIZE, LAPT_LR = 20, 16, 5e-5

if os.path.exists(f"{VMW_LAPT_ADAPTER_PATH}/pytorch_adapter.bin"):
    print(f"Emakhuwa LAPT adapter available: {VMW_LAPT_ADAPTER_PATH}")
elif RUN_MODEL_TRAINING:
    print("Loading LIACC/Emakhuwa-Monolingual ...")
    vmw_sentences = [s for s in load_dataset("LIACC/Emakhuwa-Monolingual")["train"]["sentence"]
                     if isinstance(s, str) and s.strip()]
    print(f"  {len(vmw_sentences)} sentences loaded for LAPT")

    class MLMTextDataset(Dataset):
        def __init__(self, sentences, tokenizer, max_len=MAX_LENGTH):
            self.enc = tokenizer(sentences, max_length=max_len, padding='max_length',
                                 truncation=True, return_tensors='pt')
        def __len__(self):
            return self.enc["input_ids"].size(0)
        def __getitem__(self, idx):
            return {"input_ids":      self.enc["input_ids"][idx],
                    "attention_mask": self.enc["attention_mask"][idx]}

    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True,
                                               mlm_probability=0.15)
    mlm_dl   = DataLoader(MLMTextDataset(vmw_sentences, tokenizer),
                          batch_size=LAPT_BATCH_SIZE, shuffle=True, collate_fn=collator)

    lapt_ckpt_dir   = f"{CKPT_DIR}/vmw_lapt_serengeti"
    lapt_state_path = f"{lapt_ckpt_dir}/state.json"
    lapt_last_path  = f"{lapt_ckpt_dir}/last_epoch.pt"
    os.makedirs(lapt_ckpt_dir, exist_ok=True)

    lapt_model = AutoAdapterModel.from_pretrained(MODEL_NAME, cache_dir=LOCAL_MODEL_CACHE)
    lapt_model.add_adapter(VMW_LAPT_ADAPTER_NAME, config=ADAPTER_CONFIG_LANG)
    lapt_model.add_masked_lm_head(VMW_LAPT_ADAPTER_NAME)
    lapt_model.set_active_adapters(VMW_LAPT_ADAPTER_NAME)
    lapt_model.train_adapter(VMW_LAPT_ADAPTER_NAME)
    lapt_model.to(DEVICE)

    opt    = AdamW([p for p in lapt_model.parameters() if p.requires_grad], lr=LAPT_LR)
    scaler = GradScaler()

    start_ep = 1
    if os.path.exists(lapt_state_path):
        with open(lapt_state_path) as f:
            start_ep = json.load(f)["epoch"] + 1
        lapt_model.load_state_dict(
            torch.load(lapt_last_path, map_location=DEVICE), strict=False)
        print(f"  Resuming LAPT from epoch {start_ep}")

    print(f"  Training MLM -- {len(mlm_dl)} batches/epoch x {LAPT_EPOCHS} epochs")
    for ep in range(start_ep, LAPT_EPOCHS + 1):
        lapt_model.train()
        running = 0.0
        pbar = tqdm(mlm_dl, desc=f"  LAPT epoch {ep}/{LAPT_EPOCHS}", leave=False)
        for step, batch in enumerate(pbar, 1):
            ids    = batch["input_ids"].to(DEVICE)
            mask   = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            opt.zero_grad()
            with autocast():
                out = lapt_model(input_ids=ids, attention_mask=mask, labels=labels,
                                 head=VMW_LAPT_ADAPTER_NAME)
            scaler.scale(out.loss).backward()
            scaler.step(opt); scaler.update()
            running += out.loss.item()
            pbar.set_postfix(loss=f"{running/step:.4f}")
        print(f"    epoch {ep}/{LAPT_EPOCHS} -- MLM loss: {running/len(mlm_dl):.4f}")
        torch.save(_adapter_head_state(lapt_model), lapt_last_path)
        with open(lapt_state_path, "w") as f:
            json.dump({"epoch": ep}, f)
        os.sync()

    os.makedirs(VMW_LAPT_ADAPTER_PATH, exist_ok=True)
    lapt_model.save_adapter(VMW_LAPT_ADAPTER_PATH, VMW_LAPT_ADAPTER_NAME)
    os.sync()

    print(f"Saved Emakhuwa LAPT adapter to {VMW_LAPT_ADAPTER_PATH}")
    shutil.rmtree(lapt_ckpt_dir, ignore_errors=True)
    del lapt_model; gc.collect(); torch.cuda.empty_cache()

else:
    print(
        "Emakhuwa LAPT adapter is absent. "
        "Set RUN_MODEL_TRAINING=True to create it before model training."
    )

if RUN_MODEL_TRAINING:
    if os.path.exists(RESULTS_PATH):
        try:
            with open(RESULTS_PATH) as f:
                phase3_results = json.load(f)
            print(f"Resuming -- {sum(len(v) for v in phase3_results.values())} entries saved")
        except (json.JSONDecodeError, ValueError):
            print("Corrupt results file -- starting fresh")
            phase3_results = {}
    else:
        phase3_results = {}
        print("Starting fresh (SERENGETI backbone, all African targets).")

    CONFIG_FNS = {"C1": get_c1_pool, "C2": get_c2_pool, "C3": get_c3_pool}

    for target in THIS_TIER_TARGETS:
        phase3_results.setdefault(target, {})

        ecols   = get_emotion_cols(DATA[target]["train"])
        test_df = DATA[target]["validation"]

        for track in ["track_a", "track_c"]:
            for cfg_name, pool_fn in CONFIG_FNS.items():
                run_key = f"{track}_{cfg_name}"

                # ── Skip / inference-only / re-run guard ─────────────────
                # 1. pred + weights both present  -> skip (backfill JSON if lost)
                # 2. weights present, pred missing -> inference-only, no retraining
                # 3. pred present, weights missing -> retrain to recover weights
                # 4. neither                       -> train from scratch
                _pred_path    = f"{PRED_DIR}/phase3srng_validation_{target}_{run_key}.json"
                _adapter_path = f"{ADAPTER_DIR}/{target}_{run_key}"
                _pred_ok      = os.path.exists(_pred_path)
                _adapter_ok   = os.path.exists(f"{_adapter_path}/lang/pytorch_adapter.bin")

                if _pred_ok and _adapter_ok:
                    if run_key not in phase3_results[target]:
                        _yt, _yp, _ = load_align_prediction(_pred_path, rewrite=False)
                        phase3_results[target][run_key] = score_saved_prediction(_yt, _yp, target)
                        _atomic_json_write(RESULTS_PATH, phase3_results)
                    print(f"  Skipping {LANGUAGES[target]['name']} / {run_key} -- predictions + adapters intact")
                    continue

                elif _adapter_ok and not _pred_ok:
                    print(f"  Inference-only {LANGUAGES[target]['name']} / {run_key} -- adapter found, generating prediction")
                    _lang_inf = f"lang_{target}_{run_key}_inf"
                    _m = AutoAdapterModel.from_pretrained(MODEL_NAME, cache_dir=LOCAL_MODEL_CACHE)
                    _m.load_adapter(f"{_adapter_path}/lang", load_as=_lang_inf)
                    _m.load_adapter(f"{_adapter_path}/task", load_as=TASK_ADAPTER_NAME)
                    _m.load_head(f"{_adapter_path}/head")
                    _m = _m.to(DEVICE)
                    _stack = ac.Stack(_lang_inf, TASK_ADAPTER_NAME)
                    _test_dl = DataLoader(
                        EmotionDataset(test_df, tokenizer, ecols),
                        batch_size=BATCH_SIZE * 2)
                    _preds_list, _true_list = [], []
                    with torch.no_grad():
                        with adapters.AdapterSetup(_stack):
                            for _batch in _test_dl:
                                _ids  = _batch["input_ids"].to(DEVICE)
                                _mask = _batch["attention_mask"].to(DEVICE)
                                _out  = _m(input_ids=_ids, attention_mask=_mask,
                                           head=TASK_ADAPTER_NAME)
                                _preds_list.append(
                                    (torch.sigmoid(_out.logits) > 0.5).int().cpu().numpy())
                                _true_list.append(_batch["labels"].int().numpy())
                    _y_true = np.vstack(_true_list)
                    _y_pred = np.vstack(_preds_list)
                    save_predictions("phase3srng_validation", target, run_key,
                                     _y_true, _y_pred, ecols)
                    _res = macro_f1(_y_true, _y_pred, ecols)
                    phase3_results[target][run_key] = _res
                    _atomic_json_write(RESULTS_PATH, phase3_results)
                    print(f"    macro F1 = {_res['macro_f1']:.4f}")
                    del _m; gc.collect(); torch.cuda.empty_cache()
                    continue

                elif _pred_ok and not _adapter_ok:
                    print(f"  Re-running {LANGUAGES[target]['name']} / {run_key} -- adapter weights missing from Drive")
                    phase3_results[target].pop(run_key, None)

                source_pool = pool_fn(target)

                if not source_pool and cfg_name != "C1":
                    phase3_results[target][run_key] = {
                        "macro_f1": None,
                        "note": "N/A -- empty family/genus pool (no Arabic data used; "
                                "Hausa is the only Afroasiatic BRIGHTER target)"}
                    _atomic_json_write(RESULTS_PATH, phase3_results)
                    print(f"  N/A (empty pool): {LANGUAGES[target]['name']} / {run_key}")
                    continue

                if cfg_name == "C3":
                    c2_key = f"{track}_C2"
                    if (source_pool and set(source_pool) == set(get_c2_pool(target))
                            and c2_key in phase3_results[target]):
                        phase3_results[target][run_key] = {
                            **phase3_results[target][c2_key], "aliased_from": c2_key}
                        _atomic_json_write(RESULTS_PATH, phase3_results)
                        print(f"  Aliased {LANGUAGES[target]['name']} / {run_key} -> {c2_key}")
                        continue

                train_langs = list(source_pool) + ([target] if track == "track_a" else [])

                valid_train = [c for c in train_langs if c in DATA and 'train' in DATA[c]]
                missing     = sorted(set(train_langs) - set(valid_train))
                if missing:
                    print(f"      Skipping {missing} -- no train split")
                if not valid_train:
                    phase3_results[target][run_key] = {
                        "macro_f1": None, "note": "N/A -- no source data"}
                    _atomic_json_write(RESULTS_PATH, phase3_results)
                    print(f"  N/A: {LANGUAGES[target]['name']} / {run_key}")
                    continue

                print(f"\n  [{target.upper()}] {LANGUAGES[target]['name']}"
                      f" -- {run_key} -- sources: {valid_train}")

                train_df = pd.concat([DATA[c]["train"] for c in valid_train], ignore_index=True)
                dev_df   = pd.concat([DATA[c]["validation"] for c in valid_train
                                      if "validation" in DATA[c]], ignore_index=True)

                lapt_path = None
                if target == "vmw":
                    _candidate = f"{P3_DIR}/lang_vmw_lapt_serengeti"
                    if os.path.exists(f"{_candidate}/pytorch_adapter.bin"):
                        lapt_path = _candidate

                _run_seed = RANDOM_SEED + hash(f"{target}_{run_key}") % 10000
                random.seed(_run_seed); np.random.seed(_run_seed)
                torch.manual_seed(_run_seed); torch.cuda.manual_seed_all(_run_seed)

                lang_adapter_name = f"lang_{target}_{cfg_name}_serengeti"
                model = build_madx_model(len(ecols), lang_adapter_name,
                                         pretrained_lang_adapter_path=lapt_path)
                model = train_madx(model, train_df, dev_df, tokenizer, ecols,
                                   ckpt_dir=f"{CKPT_DIR}/{target}_{run_key}")
                res   = evaluate_madx(model, test_df, tokenizer, ecols,
                                      run_tag=("phase3srng_validation", target, run_key))

                adir = f"{ADAPTER_DIR}/{target}_{run_key}"
                os.makedirs(adir, exist_ok=True)
                model.save_adapter(f"{adir}/lang", lang_adapter_name)
                model.save_adapter(f"{adir}/task", TASK_ADAPTER_NAME)
                model.save_head(f"{adir}/head",    TASK_ADAPTER_NAME)
                os.sync()

                phase3_results[target][run_key] = res
                _atomic_json_write(RESULTS_PATH, phase3_results)
                print(f"    Saved -- {run_key} macro F1 = {res['macro_f1']:.4f}"
                      f"  (adapters -> {adir})")

                del model; gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()

    print("\n" + "=" * 60)
    print("Phase 3 SERENGETI run complete.")
    print(f"Results: {RESULTS_PATH}")
else:
    print('Training disabled; saved SERENGETI predictions will be validated and analysed.')

Emakhuwa LAPT adapter available: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase3/lang_vmw_lapt_serengeti
Resuming -- 30 entries saved
  Inference-only Hausa / track_a_C1 -- adapter found, generating prediction


config.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

    macro F1 = 0.6974
  N/A (empty pool): Hausa / track_a_C2
  N/A (empty pool): Hausa / track_a_C3
  Inference-only Hausa / track_c_C1 -- adapter found, generating prediction


...

    macro F1 = 0.2283
  Inference-only Emakhuwa / track_a_C3 -- adapter found, generating prediction


    macro F1 = 0.2173
  Inference-only Emakhuwa / track_c_C1 -- adapter found, generating prediction


    macro F1 = 0.1024
  Inference-only Emakhuwa / track_c_C2 -- adapter found, generating prediction


    macro F1 = 0.1178
  Inference-only Emakhuwa / track_c_C3 -- adapter found, generating prediction


    macro F1 = 0.0910
  Skipping Nigerian Pidgin / track_a_C1 -- predictions + adapters intact
  Skipping Nigerian Pidgin / track_a_C2 -- predictions + adapters intact
  Skipping Nigerian Pidgin / track_a_C3 -- predictions + adapters intact
  Skipping Nigerian Pidgin / track_c_C1 -- predictions + adapters intact
  Skipping Nigerian Pidgin / track_c_C2 -- predictions + adapters intact
  Skipping Nigerian Pidgin / track_c_C3 -- predictions + adapters intact

Phase 3 SERENGETI run complete.
Results: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase3/phase3_serengeti_validation.json


In [ ]:
# Validate saved predictions and run within-backbone and cross-backbone tests.
# ─────────────────────────────────────────────────────────────────────────────

def _atomic_json_write_safe(path, payload):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = f"{path}.tmp"
    with open(tmp, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
        handle.flush(); os.fsync(handle.fileno())
    os.replace(tmp, path)


def load_align_prediction(path, rewrite=True):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)

    required = {"emotions", "y_true", "y_pred"}
    missing = required - set(payload)

    if missing:
        raise ValueError(f"{path}: missing keys {sorted(missing)}")
    emotions = list(payload["emotions"])

    if len(emotions) != len(set(emotions)):
        raise ValueError(f"{path}: duplicated emotion names {emotions}")
    if set(emotions) != set(EMOTION_ORDER):
        raise ValueError(f"{path}: expected {EMOTION_ORDER}, found {emotions}")

    y_true = np.asarray(payload["y_true"], dtype=int)
    y_pred = np.asarray(payload["y_pred"], dtype=int)

    if y_true.ndim != 2 or y_pred.ndim != 2 or y_true.shape != y_pred.shape:
        raise ValueError(f"{path}: incompatible arrays y_true={y_true.shape}, y_pred={y_pred.shape}")
    if y_true.shape[1] != len(emotions):
        raise ValueError(f"{path}: columns do not match stored emotion names")

    indices = [emotions.index(e) for e in EMOTION_ORDER]
    y_true = y_true[:, indices]
    y_pred = y_pred[:, indices]
    changed = emotions != EMOTION_ORDER

    if changed and rewrite:
        _atomic_json_write_safe(path, {
            "emotions": EMOTION_ORDER,
            "y_true": y_true.tolist(),
            "y_pred": y_pred.tolist(),
        })
        print(f"Aligned prediction: {os.path.basename(path)}")
    return y_true, y_pred, list(EMOTION_ORDER)


def score_saved_prediction(y_true, y_pred, lang_code):
    active = [e for e in EMOTION_ORDER if not (lang_code == "eng" and e == "disgust")]
    output = {}
    unrounded = {}
    for index, emotion in enumerate(EMOTION_ORDER):
        value = f1_score(y_true[:, index], y_pred[:, index], zero_division=0)
        unrounded[emotion] = float(value)
        output[emotion] = round(float(value), 4)
    output["macro_f1"] = round(float(np.mean([unrounded[e] for e in active])), 4)
    return output

from itertools import combinations
import zlib

N_BOOT = 10_000
BOOT_CHUNK = 64
BASE_SEED = 42
ALPHA = 0.05


def _stable_seed(label):
    return int((BASE_SEED + zlib.crc32(label.encode("utf-8"))) % (2**32 - 1))


def _active_indices(lang):
    return [i for i,e in enumerate(EMOTION_ORDER) if not (lang == "eng" and e == "disgust")]


def _bootstrap_scores(y_true, systems, lang, seed_label):
    active = _active_indices(lang)
    y = np.asarray(y_true[:, active], dtype=np.int8)
    names = list(systems)
    blocks = []
    for name in names:
        pred = np.asarray(systems[name][:, active], dtype=np.int8)
        if pred.shape != y.shape:
            raise ValueError(f"{seed_label}/{name}: {pred.shape} != {y.shape}")
        tp = ((y == 1) & (pred == 1)).astype(np.uint8)
        fp = ((y == 0) & (pred == 1)).astype(np.uint8)
        fn = ((y == 1) & (pred == 0)).astype(np.uint8)
        blocks.append(np.stack([tp, fp, fn], axis=2))
    contributions = np.stack(blocks, axis=1)
    flat = contributions.reshape(len(y), -1)
    n_systems, n_labels = len(names), len(active)

    def _macro(counts):
        tp, fp, fn = counts[...,0], counts[...,1], counts[...,2]
        den = 2*tp + fp + fn
        f1 = np.divide(2*tp, den, out=np.zeros_like(den,dtype=float), where=den!=0)
        return np.mean(f1, axis=-1)

    obs_counts = flat.sum(axis=0,dtype=np.int64).reshape(n_systems,n_labels,3)
    obs_values = _macro(obs_counts)
    observed = {n: float(obs_values[i]) for i,n in enumerate(names)}
    boot_matrix = np.empty((N_BOOT,n_systems),dtype=float)
    rng = np.random.default_rng(_stable_seed(seed_label))
    pos=0

    while pos < N_BOOT:
        size=min(BOOT_CHUNK,N_BOOT-pos)
        idx=rng.integers(0,len(y),size=(size,len(y)),dtype=np.int32)
        counts=flat[idx].sum(axis=1,dtype=np.int64).reshape(size,n_systems,n_labels,3)
        boot_matrix[pos:pos+size]=_macro(counts)
        pos += size
    return observed,{n:boot_matrix[:,i] for i,n in enumerate(names)}


def _summarise(values, observed):
    values=np.asarray(values,dtype=float)
    p=min(2*min((np.sum(values<=0)+1)/(len(values)+1),
                (np.sum(values>=0)+1)/(len(values)+1)),1.0)
    lo,hi=np.percentile(values,[2.5,97.5])
    return {"delta":float(observed),"ci_low":float(lo),"ci_high":float(hi),"p":float(p)}


def _holm(pvalues):
    p=np.asarray(pvalues,dtype=float); m=len(p)
    order=np.argsort(p); adjusted=np.empty(m); running=0.0
    for rank,idx in enumerate(order):
        running=max(running,min(1.0,(m-rank)*p[idx])); adjusted[idx]=running
    return adjusted


def _finish_family(rows, expected, output_path, family):
    frame = pd.DataFrame(rows)
    if len(frame) != expected:
        raise ValueError(f"{family}: expected {expected} tests, found {len(frame)}")
    if frame.empty:
        print(f"  {family}: no pairwise tests to run yet -- skipping (fewer than 2 configs complete per language/track)")
        return frame
    frame["Holm-adjusted p-value"] = _holm(frame["Raw p-value"].astype(float).values)
    frame["Significant after Holm correction (alpha=0.05)"] = frame["Holm-adjusted p-value"] < ALPHA
    frame["Evaluation split"] = "Validation"
    frame["Statistical test"] = "Paired bootstrap; Holm correction applied within the stated comparison family"
    frame.to_csv(output_path, index=False)
    print(f"Saved {len(frame)} tests: {output_path}")
    return frame

RESULTS_PATH=f"{P3_DIR}/phase3_serengeti_validation.json"

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH,encoding="utf-8") as _f: corrected_results=json.load(_f)
else: corrected_results={}

SERENGETI_LANGS=["hau","kin","yor","vmw","pcm"]

for lang in SERENGETI_LANGS:
    corrected_results.setdefault(lang,{})
    for track in ["a","c"]:
        for cfg in ["C1","C2","C3"]:
            path=f"{PRED_DIR}/phase3srng_validation_{lang}_track_{track}_{cfg}.json"
            if not os.path.exists(path): continue
            y,p,_=load_align_prediction(path,rewrite=True)
            key=f"track_{track}_{cfg}"; prior=corrected_results[lang].get(key,{})
            score=score_saved_prediction(y,p,lang)
            corrected_results[lang][key]={**prior,**score} if isinstance(prior,dict) else score

_atomic_json_write_safe(RESULTS_PATH,corrected_results)
print(f"Validated Phase 3 SERENGETI results: {RESULTS_PATH}")

rows=[]
for lang in SERENGETI_LANGS:
    for track in ["a","c"]:
        systems={}; reference=None
        for cfg in ["C1","C2","C3"]:
            path=f"{PRED_DIR}/phase3srng_validation_{lang}_track_{track}_{cfg}.json"
            if not os.path.exists(path): continue
            y,p,_=load_align_prediction(path,rewrite=False)
            if reference is None: reference=y
            elif not np.array_equal(reference,y): raise ValueError(f"{lang}/{track}/{cfg}: gold mismatch")
            systems[cfg]=p

        if len(systems)<2: continue
        observed,boot=_bootstrap_scores(reference,systems,lang,f"phase3_srng_{lang}_{track}")

        for a,b in combinations(["C1","C2","C3"],2):
            if a not in systems or b not in systems: continue
            stat=_summarise(boot[a]-boot[b],observed[a]-observed[b])
            rows.append({
                "Language code":                              lang.upper(),
                "Language":                                   LANGUAGES[lang]["name"],
                "Evaluation track":                           f"Track {track.upper()}",
                "Comparison":                                 f"{a} vs {b}",
                "First transfer configuration":               a,
                "Second transfer configuration":              b,
                "First-condition macro-F1":                   observed[a],
                "Second-condition macro-F1":                  observed[b],
                "Macro-F1 difference (first minus second)":   stat["delta"],
                "95% confidence interval lower bound":        stat["ci_low"],
                "95% confidence interval upper bound":        stat["ci_high"],
                "Raw p-value":                                stat["p"],
                "Higher-scoring condition":                   a if stat["delta"] > 0 else b,
            })

phase3_srng_significance=_finish_family(rows,len(rows),f"{P3_DIR}/phase3_serengeti_validation_significance.csv","Phase 3 SERENGETI")

Validated Phase 3 SERENGETI results: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase3/phase3_serengeti_validation.json
Saved 24 tests: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase3/phase3_serengeti_validation_significance.csv


In [ ]:
# Direct matched-backbone comparisons from saved predictions.
bb=[]

for lang in SERENGETI_LANGS:
    for track in ["a","c"]:
        for cfg in ["C1","C2","C3"]:
            xp=f"{PRED_DIR}/phase3xlmrW_validation_{lang}_track_{track}_{cfg}.json"
            sp=f"{PRED_DIR}/phase3srng_validation_{lang}_track_{track}_{cfg}.json"

            if not (os.path.exists(xp) and os.path.exists(sp)): continue

            xy,xp_,_=load_align_prediction(xp,False); sy,sp_,_=load_align_prediction(sp,False)

            if not np.array_equal(xy,sy): raise ValueError(f"backbone/{lang}/{track}/{cfg}: gold mismatch")

            observed,boot=_bootstrap_scores(xy,{"xlmr":xp_,"serengeti":sp_},lang,f"backbone_{lang}_{track}_{cfg}")
            stat=_summarise(boot["serengeti"]-boot["xlmr"],observed["serengeti"]-observed["xlmr"])

            bb.append({
                "Language code":                              lang.upper(),
                "Language":                                   LANGUAGES[lang]["name"],
                "Evaluation track":                           f"Track {track.upper()}",
                "Transfer configuration":                     cfg,
                "XLM-R macro-F1":                             observed["xlmr"],
                "SERENGETI macro-F1":                          observed["serengeti"],
                "Backbone effect (SERENGETI minus XLM-R)":    stat["delta"],
                "95% confidence interval lower bound":        stat["ci_low"],
                "95% confidence interval upper bound":        stat["ci_high"],
                "Raw p-value":                                stat["p"],
                "Higher-scoring condition":                   "SERENGETI" if stat["delta"] > 0 else "XLM-R",
            })

phase3_backbone_significance=_finish_family(bb,len(bb),f"{P3_DIR}/phase3_backbone_validation_significance.csv","Direct backbone")

Saved 25 tests: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase3/phase3_backbone_validation_significance.csv


In [ ]:
# Descriptive Phase 3 comparison table
def _safe_load(path: str, label: str = '') -> dict:
    if not os.path.exists(path):
        print(f"  Not found{' (' + label + ')' if label else ''}: {path}")
        return {}
    with open(path) as f:
        return json.load(f)

xlmr_w  = _safe_load(f"{P3_DIR}/phase3_xlmr_validation_scores.json", "XLM-R P3 matched recipe")
srng_p3 = _safe_load(RESULTS_PATH,                          "SERENGETI P3")

_p1_srng = _safe_load(f"{SAVE_DIR}/Phase1/serengeti/phase1_serengeti.json", "SRNG B2")
srng_b2  = _p1_srng.get("track_a", {})

B4_A = {"eng": 0.823, "hin": 0.926, "rus": 0.901, "hau": 0.751, "kin": 0.657,
       "sun": 0.550, "yor": 0.461, "vmw": 0.325, "pcm": 0.674}
B4_C = {"eng": 0.797, "hin": 0.919, "rus": 0.906, "hau": 0.709, "kin": 0.519,
       "sun": 0.467, "yor": 0.359, "vmw": 0.210, "pcm": 0.674}

LANG_NAME = {c: LANGUAGES[c]['name'] for c in THIS_TIER_TARGETS}
CONFIGS   = ["C1", "C2", "C3"]

rows = []
for code in THIS_TIER_TARGETS:
    for cfg in CONFIGS:
        for track, label in [("track_a", "A"), ("track_c", "C")]:
            run_key = f"{track}_{cfg}"
            rows.append({
                'Language':                                                        LANG_NAME[code],
                'Language code':                                                   code.upper(),
                'Transfer configuration':                                          cfg,
                'Evaluation track':                                                f'Track {label}',
                'XLM-R macro-F1 (matched setup)':                                 xlmr_w.get(code, {}).get(run_key, {}).get('macro_f1'),
                'SERENGETI macro-F1':                                              srng_p3.get(code, {}).get(run_key, {}).get('macro_f1'),
                'Phase 1 SERENGETI reference macro-F1':                           srng_b2.get(code, {}).get('macro_f1') if label == 'A' else None,
                'Evaluation split':                                                'Validation',
                'Published Track A test best macro-F1 (context only)':            B4_A.get(code),
                'Published Track C test best macro-F1 (context only)':            B4_C.get(code),
                "Published test best for this row's track (context only)":       B4_A.get(code) if label == 'A' else B4_C.get(code),
                'Published benchmark source':                                      BENCHMARK_SRC_A if label == 'A' else BENCHMARK_SRC_C,
                'Benchmark comparison status':                                     PHASE3_BENCHMARK_STATUS,
            })

df_cmp = pd.DataFrame(rows)
print(df_cmp.to_string(index=False))

out_path = f"{P3_DIR}/phase3_serengeti_validation_comparison.csv"

df_cmp.to_csv(out_path, index=False)
print(f"\nComparison saved to {out_path}")

       Language Language code Transfer configuration Evaluation track  XLM-R macro-F1 (matched setup)  SERENGETI macro-F1  Phase 1 SERENGETI reference macro-F1 Evaluation split  Published Track A test best macro-F1 (context only)  Published Track C test best macro-F1 (context only)  Published test best for this row's track (context only)              Published benchmark source                                          Benchmark comparison status
          Hausa           HAU                     C1          Track A                          0.6792              0.6974                                0.6966       Validation                                                0.751                                                0.709                                                    0.751 SemEval-2025 Task 11, Table 5 (Track A) Context only: model score is validation; published reference is test
          Hausa           HAU                     C1          Track C                          0.3016 